# Column Transformer

In [1]:
import numpy as np
import pandas as pd

In [2]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder

In [3]:
df = pd.read_csv('covid_toy.csv')

In [4]:
df.head()

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [5]:
df['cough'].value_counts()

cough
Mild      62
Strong    38
Name: count, dtype: int64

In [6]:
df['city'].value_counts()

city
Kolkata      32
Bangalore    30
Delhi        22
Mumbai       16
Name: count, dtype: int64

In [8]:
df.isnull().sum()

age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64

In [10]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(df.drop(columns=['has_covid']), df['has_covid'], test_size=0.2, random_state=42)

In [11]:
X_train.head()

,age,gender,fever,cough,city
55,81,Female,101.0,Mild,Mumbai
88,5,Female,100.0,Mild,Kolkata
26,19,Female,100.0,Mild,Kolkata
42,27,Male,100.0,Mild,Delhi
69,73,Female,103.0,Mild,Delhi


### Manual Implementation

In [18]:
# Handling fever column missing values

si = SimpleImputer()
X_train_fever = si.fit_transform(X_train[['fever']])
X_test_fever = si.transform(X_test[['fever']])

In [30]:
X_train_fever.shape, X_test_fever.shape

((80, 1), (20, 1))

In [19]:
# Ordinal Encoding -> cough
oe = OrdinalEncoder(categories=[['Mild', 'Strong']])

X_train_cough = oe.fit_transform(X_train[['cough']])
X_test_cough = oe.transform(X_test[['cough']])

In [31]:
X_train_cough.shape, X_test_cough.shape

((80, 1), (20, 1))

In [42]:
# OneHotEncoding -> Gender, City
ohe = OneHotEncoder(drop='first')

X_train_gender_city = ohe.fit_transform(X_train[['gender', 'city']]).toarray()
X_test_gender_city = ohe.transform(X_test[['gender', 'city']]).toarray()

In [43]:
X_train_gender_city.shape, X_test_gender_city.shape

((80, 4), (20, 4))

In [44]:
# Extracting Age

X_train_age = X_train.drop(columns=['gender', 'fever', 'cough', 'city']).values
X_test_age = X_test.drop(columns=['gender', 'fever', 'cough', 'city']).values

In [45]:
X_train_age.shape, X_test_age.shape

((80, 1), (20, 1))

In [46]:
X_train_transformed = np.concatenate((X_train_age, X_train_fever, X_train_gender_city, X_train_cough), axis=1)

In [47]:
X_test_transformed = np.concatenate((X_test_age, X_test_fever, X_test_gender_city, X_test_cough), axis=1)

In [49]:
X_train_transformed.shape, X_test_transformed.shape

((80, 7), (20, 7))

### Column Transformer Pipeline

In [50]:
from sklearn.compose import ColumnTransformer

In [53]:
transformer = ColumnTransformer(transformers=[
    ('tnf1', SimpleImputer(), ['fever']),
    ('tnf2', OrdinalEncoder(categories=[['Mild', 'Strong']]), ['cough']),
    ('tnf3', OneHotEncoder(sparse_output=False, drop='first'), ['gender', 'city']),
], remainder='passthrough')

In [55]:
X_train_col_transformed = transformer.fit_transform(X_train)

In [56]:
X_test_col_transformed = transformer.transform(X_test)

In [57]:
X_train_col_transformed.shape, X_test_col_transformed.shape

((80, 7), (20, 7))